# Lab · Dataset và transform · LỜI GIẢI

> Chỉ mở file này khi bạn đã thử bài tập, hoặc đã dùng hết cả 3 gợi ý.

Bài tập cho [`lesson.md`](../lesson.md) (Module 2). Theo đúng hai lab của khóa: **Phần A** viết dataset và transform, **Phần B** dùng dataset dựng sẵn MNIST của `torchvision`.

**Cách làm:** tự thử trước. Kẹt thì mở **Gợi ý 1**, rồi **Gợi ý 2**, cuối cùng mới **Gợi ý 3**. Lời giải ở thư mục `solution/`.

Môi trường do **uv** quản lý: trong thư mục khóa `04-intro-neural-networks-pytorch/`, chạy `uv sync` (một lần), rồi `uv run jupyter lab`.
Phần B tự tải MNIST (~65 MB) vào thư mục `data/` của khóa ở lần chạy đầu, cần có mạng.

## Chuẩn bị

In [ ]:
import torch
from torch.utils.data import Dataset
import torchvision.datasets as dsets
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
%matplotlib inline

torch.manual_seed(0)
DATA_DIR = "../../../data"   # MNIST được tải về đây
print("PyTorch", torch.__version__)

## Bài 1. Phần A · Viết dataset `toy_set`

Viết lớp `toy_set` kế thừa `Dataset`:
- `__init__(self, length=100, transform=None)`: lưu `self.len = length`; `self.x` là tensor shape `(length, 2)` toàn số **2**; `self.y` là tensor shape `(length, 1)` toàn số **1**; lưu `self.transform`.
- `__len__`: trả số mẫu.
- `__getitem__(self, index)`: tạo mẫu `(x[index], y[index])`; nếu có `transform` thì áp lên mẫu; trả về mẫu.

Sau đó tạo `ds = toy_set()`, rồi lưu `n = len(ds)` và `first = ds[0]`.

In [ ]:
class toy_set(Dataset):
    def __init__(self, length=100, transform=None):
        self.len = length
        self.x = 2 * torch.ones(length, 2)
        self.y = torch.ones(length, 1)
        self.transform = transform

    def __getitem__(self, index):
        sample = self.x[index], self.y[index]
        if self.transform:
            sample = self.transform(sample)
        return sample

    def __len__(self):
        return self.len

ds = toy_set()
n = len(ds)
first = ds[0]
print(n, first)

In [ ]:
assert n == 100
assert isinstance(first, tuple) and len(first) == 2, "mẫu phải là tuple (x, y)"
assert first[0].tolist() == [2.0, 2.0] and first[1].tolist() == [1.0]
print("✅ Bài 1 đúng")

## Bài 2. Duyệt dataset bằng vòng `for`

- `first3`: list gồm 3 mẫu đầu, lấy bằng vòng `for` trên `range(3)` và index `ds[i]`.
- `all_samples`: list gồm **mọi** mẫu của `ds`, lấy bằng cách duyệt trực tiếp trên `ds` (không dùng index).

Câu hỏi suy nghĩ: vì sao duyệt thẳng trên `ds` chạy được và dừng đúng chỗ?

In [ ]:
first3 = [ds[i] for i in range(3)]
all_samples = list(ds)
print(len(first3), len(all_samples))

In [ ]:
assert len(first3) == 3
assert len(all_samples) == len(ds) == 100
print("✅ Bài 2 đúng")

## Bài 3. Luyện tập: dataset dài 50

Tạo `ds50` là một `toy_set` có **50** mẫu và lưu độ dài vào `n50`.

In [ ]:
ds50 = toy_set(length=50)
n50 = len(ds50)
print(n50)

In [ ]:
assert n50 == 50
print("✅ Bài 3 đúng")

## Bài 4. Transform `add_mult`

Viết lớp transform `add_mult` với `__init__(self, addx=1, muly=2)` và `__call__(self, sample)`:
nhận mẫu `(x, y)`, trả về `(x + addx, y * muly)`.

Tạo `a_m = add_mult()`, rồi `transformed = a_m(ds[0])`. **Đoán trước** kết quả.

In [ ]:
class add_mult:
    def __init__(self, addx=1, muly=2):
        self.addx = addx
        self.muly = muly

    def __call__(self, sample):
        x, y = sample
        x = x + self.addx
        y = y * self.muly
        return x, y

a_m = add_mult()
transformed = a_m(ds[0])
print(transformed)

In [ ]:
assert transformed[0].tolist() == [3.0, 3.0], "x phải được cộng 1: [2, 2] → [3, 3]"
assert transformed[1].tolist() == [2.0], "y phải được nhân 2: [1] → [2]"
assert ds[0][0].tolist() == [2.0, 2.0], "dữ liệu gốc không được đổi"
print("✅ Bài 4 đúng")

## Bài 5. Gắn transform vào dataset

Tạo `cust` là một `toy_set` có `transform=add_mult()`.
Lưu `cust[0]` vào `cust_first` và `ds[0]` vào `orig_first`.
Câu hỏi: dataset gốc `ds` có bị thay đổi không?

In [ ]:
cust = toy_set(transform=add_mult())
cust_first = cust[0]
orig_first = ds[0]
print(cust_first, orig_first)

In [ ]:
assert cust_first[0].tolist() == [3.0, 3.0] and cust_first[1].tolist() == [2.0]
assert orig_first[0].tolist() == [2.0, 2.0] and orig_first[1].tolist() == [1.0]
print("✅ Bài 5 đúng: transform chỉ tác động khi lấy mẫu, dữ liệu gốc giữ nguyên")

## Bài 6. Luyện tập: tự viết `my_add_mult`

Viết transform `my_add_mult` của riêng bạn: **trừ 1** khỏi `x` và **nhân y với 3**.
Tạo `mine = toy_set(transform=my_add_mult())` và lưu 3 mẫu đầu vào `mine3` (list).

In [ ]:
class my_add_mult:
    def __call__(self, sample):
        x, y = sample
        return x - 1, y * 3

mine = toy_set(transform=my_add_mult())
mine3 = [mine[i] for i in range(3)]
print(mine3)

In [ ]:
assert len(mine3) == 3
for x, y in mine3:
    assert x.tolist() == [1.0, 1.0] and y.tolist() == [3.0]
print("✅ Bài 6 đúng")

## Bài 7. `Compose`: ghép transform

Viết transform `mult` (tham số `mult=100`) nhân **cả `x` lẫn `y`** với số đó.
Rồi `data_transform = transforms.Compose([add_mult(), mult()])` và `compose_ds = toy_set(transform=data_transform)`.
Lưu `compose_ds[0]` vào `comp_first`. **Đoán trước** `x` và `y` bằng bao nhiêu (mẫu gốc `x=[2,2]`, `y=[1]`).

In [ ]:
class mult:
    def __init__(self, mult=100):
        self.mult = mult

    def __call__(self, sample):
        x, y = sample
        return x * self.mult, y * self.mult

data_transform = transforms.Compose([add_mult(), mult()])
compose_ds = toy_set(transform=data_transform)
comp_first = compose_ds[0]
print(comp_first)

In [ ]:
assert comp_first[0].tolist() == [300.0, 300.0], "(2 + 1) × 100 = 300"
assert comp_first[1].tolist() == [200.0], "1 × 2 × 100 = 200"
print("✅ Bài 7 đúng")

## Bài 8. Luyện tập: đảo thứ tự Compose

Tạo pipeline mà `mult()` chạy **trước**, `add_mult()` chạy sau. Gắn vào `toy_set` mới và lưu mẫu đầu vào `rev_first`.
**Đoán trước:** `x` lần này bằng bao nhiêu, và vì sao khác bài 7?

In [ ]:
rev_ds = toy_set(transform=transforms.Compose([mult(), add_mult()]))
rev_first = rev_ds[0]
print(rev_first)

In [ ]:
assert rev_first[0].tolist() == [201.0, 201.0], "2 × 100 + 1 = 201"
assert rev_first[1].tolist() == [200.0]
print("✅ Bài 8 đúng: thứ tự transform ảnh hưởng kết quả")

## Bài 9. Phần B · Dataset MNIST dựng sẵn

Tải MNIST bằng `torchvision.datasets.MNIST`: `root=DATA_DIR`, `download=True`, `transform=transforms.ToTensor()`. Lưu vào `mnist`.
Sau đó kiểm tra: `type_first` (kiểu của phần tử đầu), `len_tuple` (độ dài tuple đó), `img_shape` (shape của ảnh), `label0` (nhãn mẫu đầu).

Câu hỏi: vì sao ảnh có shape `(1, 28, 28)` mà không phải `(28, 28)`?

In [ ]:
mnist = dsets.MNIST(root=DATA_DIR, download=True, transform=transforms.ToTensor())
img, label = mnist[0]
type_first = type(mnist[0])
len_tuple = len(mnist[0])
img_shape = tuple(img.shape)
label0 = label
print(len(mnist), type_first, len_tuple, img_shape, label0)

In [ ]:
assert len(mnist) == 60000
assert type_first is tuple and len_tuple == 2
assert img_shape == (1, 28, 28)
assert label0 == 5, "mẫu đầu của MNIST train là chữ số 5"
print("✅ Bài 9 đúng")

## Bài 10. Hiển thị ảnh

Viết hàm `show_data(sample, shape=(28, 28))` nhận một mẫu `(ảnh, nhãn)`, vẽ ảnh bằng `plt.imshow(..., cmap="gray")` và đặt tiêu đề `y = <nhãn>`.
Gọi nó cho `mnist[0]` và `mnist[1]`. (Mẫu thứ hai là chữ số 0.)

In [ ]:
def show_data(sample, shape=(28, 28)):
    plt.imshow(sample[0].numpy().reshape(shape), cmap="gray")
    plt.title("y = " + str(sample[1]))

show_data(mnist[0])
plt.show()
show_data(mnist[1])
plt.show()

In [ ]:
show_data(mnist[0])
assert plt.gca().get_title() == "y = 5", "tiêu đề phải là 'y = 5'"
plt.close("all")
assert mnist[1][1] == 0
print("✅ Bài 10 đúng")

## Bài 11. Transform ảnh: cắt giữa

Tạo `crop_ds`: MNIST với transform ghép **`CenterCrop(20)` rồi `ToTensor()`**. Lưu shape của ảnh đầu vào `crop_shape`.
Vẽ thử ảnh đầu bằng `show_data(crop_ds[0], shape=(20, 20))`.

In [ ]:
crop_tf = transforms.Compose([transforms.CenterCrop(20), transforms.ToTensor()])
crop_ds = dsets.MNIST(root=DATA_DIR, download=True, transform=crop_tf)
crop_shape = tuple(crop_ds[0][0].shape)
show_data(crop_ds[0], shape=(20, 20))
plt.show()
print(crop_shape)

In [ ]:
assert crop_shape == (1, 20, 20)
print("✅ Bài 11 đúng")

## Bài 12. Transform ảnh: lật ngang

Tạo `flip_ds`: MNIST với `RandomHorizontalFlip(p=1)` rồi `ToTensor()`. Với `p=1` ảnh **luôn** được lật.
Kiểm tra bằng cách so sánh `flip_ds[0][0]` với ảnh gốc `mnist[0][0]` lật theo chiều ngang (`torch.flip`): lưu kết quả so sánh (`True/False`) vào `is_flipped`.

In [ ]:
flip_tf = transforms.Compose([transforms.RandomHorizontalFlip(p=1), transforms.ToTensor()])
flip_ds = dsets.MNIST(root=DATA_DIR, download=True, transform=flip_tf)
is_flipped = torch.equal(flip_ds[0][0], torch.flip(mnist[0][0], dims=[2]))
show_data(flip_ds[0])
plt.show()
print(is_flipped)

In [ ]:
assert is_flipped is True
print("✅ Bài 12 đúng")

## Bài 13. Luyện tập: lật dọc rồi lật ngang

Tạo `vh_ds` với pipeline: `RandomVerticalFlip(p=1)`, `RandomHorizontalFlip(p=1)`, rồi `ToTensor()`.
Lưu `vh_ok = True` nếu ảnh đầu của nó bằng ảnh gốc `mnist[0][0]` lật theo **cả hai** chiều cao và rộng.

In [ ]:
vh_tf = transforms.Compose([
    transforms.RandomVerticalFlip(p=1),
    transforms.RandomHorizontalFlip(p=1),
    transforms.ToTensor(),
])
vh_ds = dsets.MNIST(root=DATA_DIR, download=True, transform=vh_tf)
vh_ok = torch.equal(vh_ds[0][0], torch.flip(mnist[0][0], dims=[1, 2]))
show_data(vh_ds[0])
plt.show()
print(vh_ok)

In [ ]:
assert vh_ok is True
print("✅ Bài 13 đúng")

## Hết